# VitalDB reduced-model Colab GPU validation

This notebook performs environment and data validation, tests, two validation-only smoke runs, and a short CUDA benchmark. It does not run full scientific training, group retraining, feature selection, RL, or test-set importance analysis. Smoke attention is pipeline diagnostics only.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = DRIVE_PROJECT_ROOT / 'data/modeling/full'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs'
ENV_OUTPUT_DIR = OUTPUT_ROOT / 'colab_environment'
GRU_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/gru/seed_42'
ATTENTION_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/attention/seed_42'
BENCHMARK_DIR = OUTPUT_ROOT / 'runtime_benchmark'
for path in (ENV_OUTPUT_DIR, GRU_SMOKE_DIR.parent, ATTENTION_SMOKE_DIR.parent, BENCHMARK_DIR):
    path.mkdir(parents=True, exist_ok=True)
print({'dataset': str(DATASET_DIR), 'outputs': str(OUTPUT_ROOT)})

{'dataset': '/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full', 'outputs': '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs'}


In [5]:
import subprocess

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Active git commit:', commit)

Active git commit: ab30d63a347ecd3d3f1d2de3ef26b0b0569abf90


## Preserve Colab CUDA PyTorch
The dependency script installs only missing non-PyTorch packages. It rejects `torch`, `torchvision`, or `torchaudio` in the Colab requirement list and verifies that the PyTorch version did not change.

In [6]:
import os
import sys
import torch

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.colab_workflow import run_streaming_command
print('Before dependency check:', torch.__version__, torch.version.cuda, torch.cuda.is_available(), flush=True)
run_streaming_command([sys.executable, 'scripts/install_colab_dependencies.py'], stage='Colab dependency installation', cwd=REPO_DIR)

Before dependency check: 2.11.0+cu128 12.8 True


CompletedProcess(args=['/usr/bin/python3', 'scripts/install_colab_dependencies.py'], returncode=0)

In [7]:
# This writes the audit before raising if a GPU was not assigned.
run_streaming_command([
    sys.executable, 'scripts/audit_colab_environment.py',
    '--repo-dir', str(REPO_DIR),
    '--output', str(ENV_OUTPUT_DIR / 'environment_audit.json'),
], stage='Colab environment audit', cwd=REPO_DIR)

CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_colab_environment.py', '--repo-dir', '/content/VitalDB-Feature-Selection', '--output', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_environment/environment_audit.json'], returncode=0)

In [8]:
run_streaming_command([
    sys.executable, 'scripts/validate_modeling_artifacts.py',
    '--dataset-dir', str(DATASET_DIR),
    '--output', str(ENV_OUTPUT_DIR / 'data_artifact_audit.json'),
], stage='modeling artifact validation', cwd=REPO_DIR)

CompletedProcess(args=['/usr/bin/python3', 'scripts/validate_modeling_artifacts.py', '--dataset-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full', '--output', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_environment/data_artifact_audit.json'], returncode=0)

In [9]:
test_result = run_streaming_command(
    [sys.executable, '-m', 'pytest', '-q'], stage='full pytest', cwd=REPO_DIR
)
(ENV_OUTPUT_DIR / 'pytest_colab.txt').write_text(test_result.stdout, encoding='utf-8')

......................................................................   [100%]
70 passed in 11.82s



## Validation-only CUDA smoke runs
Run status markers distinguish complete and interrupted directories. Completed runs are skipped. Smoke mode uses at most two epochs and does not evaluate the test split.

In [10]:
from src.colab_workflow import inspect_run_completion

gru_status = inspect_run_completion(GRU_SMOKE_DIR, 'gru')
print(gru_status)
if not gru_status['complete']:
    run_streaming_command([
        sys.executable, 'scripts/run_baselines.py', 'gru',
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(GRU_SMOKE_DIR),
        '--exclude-dynamic-features', 'bis_error',
        '--smoke', '--seed', '42', '--device', 'cuda',
    ], stage='GRU validation smoke', cwd=REPO_DIR)
run_streaming_command([
    sys.executable, 'scripts/verify_colab_smoke.py',
    '--run-dir', str(GRU_SMOKE_DIR), '--dataset-dir', str(DATASET_DIR),
    '--model', 'gru', '--output', str(GRU_SMOKE_DIR / 'smoke_audit.json'),
], stage='GRU smoke verification', cwd=REPO_DIR)

{'run_dir': '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/gru/seed_42', 'model': 'gru', 'complete': False, 'missing_artifacts': ['run_status.json', 'config.json', 'best_model.pt', 'last_model.pt', 'training_history.csv', 'val_predictions.csv', 'val_metrics.json', 'case_metrics.csv', 'runtime.json'], 'status': 'missing', 'safe_action': 'restart_or_resume_this_directory_only'}


CompletedProcess(args=['/usr/bin/python3', 'scripts/verify_colab_smoke.py', '--run-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/gru/seed_42', '--dataset-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full', '--model', 'gru', '--output', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/gru/seed_42/smoke_audit.json'], returncode=0)

In [11]:
attention_status = inspect_run_completion(ATTENTION_SMOKE_DIR, 'attention')
print(attention_status)
if not attention_status['complete']:
    run_streaming_command([
        sys.executable, 'scripts/run_attention.py',
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(ATTENTION_SMOKE_DIR),
        '--exclude-dynamic-features', 'bis_error',
        '--smoke', '--seed', '42', '--device', 'cuda',
    ], stage='attention validation smoke', cwd=REPO_DIR)
run_streaming_command([
    sys.executable, 'scripts/verify_colab_smoke.py',
    '--run-dir', str(ATTENTION_SMOKE_DIR), '--dataset-dir', str(DATASET_DIR),
    '--model', 'attention',
    '--output', str(ATTENTION_SMOKE_DIR / 'smoke_audit.json'),
], stage='attention smoke verification', cwd=REPO_DIR)

{'run_dir': '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/attention/seed_42', 'model': 'attention', 'complete': False, 'missing_artifacts': ['run_status.json', 'config.json', 'best_model.pt', 'last_model.pt', 'training_history.csv', 'val_predictions.csv', 'val_metrics.json', 'case_metrics.csv', 'val_attention.npz', 'attention_metadata.json'], 'status': 'missing', 'safe_action': 'restart_or_resume_this_directory_only'}


CompletedProcess(args=['/usr/bin/python3', 'scripts/verify_colab_smoke.py', '--run-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/attention/seed_42', '--dataset-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full', '--model', 'attention', '--output', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/attention/seed_42/smoke_audit.json'], returncode=0)

In [12]:
run_streaming_command([
    sys.executable, 'scripts/benchmark_colab_gpu.py',
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(BENCHMARK_DIR),
    '--batch-sizes', '256,512,1024,2048',
    '--measured-batches', '20', '--warmup-batches', '3', '--seed', '42',
], stage='CUDA benchmark', cwd=REPO_DIR)

CompletedProcess(args=['/usr/bin/python3', 'scripts/benchmark_colab_gpu.py', '--dataset-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full', '--output-dir', '/content/drive/MyDrive/VitalDB-Feature-Selection/outputs/runtime_benchmark', '--batch-sizes', '256,512,1024,2048', '--measured-batches', '20', '--warmup-batches', '3', '--seed', '42'], returncode=0)

## Interpretation and backend comparability
CPU and CUDA predictions need not be bitwise identical. Compare rows, feature order, architecture, seed configuration, finite outputs, and repeated evaluation of the same checkpoint. Every future paired scientific comparison must use one device/backend for every model and seed. Do not mix CPU and GPU seeds unless the analysis is explicitly designed as backend sensitivity. The existing test set is a development test set, not a pristine final holdout. Future candidate selection remains validation-only; final claims should use previously unseen cases or another pre-specified evaluation design.

In [13]:
print('Persistent outputs:')
print('Environment:', ENV_OUTPUT_DIR)
print('GRU smoke:', GRU_SMOKE_DIR)
print('Attention smoke:', ATTENTION_SMOKE_DIR)
print('GPU benchmark:', BENCHMARK_DIR)
print('Git commit used:', commit)

Persistent outputs:
Environment: /content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_environment
GRU smoke: /content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/gru/seed_42
Attention smoke: /content/drive/MyDrive/VitalDB-Feature-Selection/outputs/colab_smoke/attention/seed_42
GPU benchmark: /content/drive/MyDrive/VitalDB-Feature-Selection/outputs/runtime_benchmark
Git commit used: ab30d63a347ecd3d3f1d2de3ef26b0b0569abf90


In [14]:
from pathlib import Path

DATASET_DIR = Path(
    "/content/drive/MyDrive/VitalDB-Feature-Selection/data/modeling/full"
)

print("exists:", DATASET_DIR.exists())
print("contents:")
for path in sorted(DATASET_DIR.iterdir()):
    print(" -", path.name)

for filename in ("train.npz", "val.npz", "test.npz"):
    assert (DATASET_DIR / filename).is_file(), f"Missing: {filename}"

print("Core NPZ files confirmed.")

exists: True
contents:
 - case_level_target_summary.csv
 - dataset_metadata.json
 - dataset_report.json
 - feature_manifest.csv
 - full_dataset_audit.json
 - preprocessing.pkl
 - preprocessing_statistics.csv
 - splits
 - test.npz
 - test_metadata.csv
 - train.npz
 - train_metadata.csv
 - val.npz
 - val_metadata.csv
Core NPZ files confirmed.
